# 0330 · Base analítica de la reserva (septiembre a noviembre), reglas v4

> **Fase 4 (prueba única).** Mismo cuaderno que `0330_BaseAnaliticaV4`, con las mismas reglas (SLA en días hábiles, red activa más reciente, T0 posterior a la admisión, abiertas al corte), apuntando a `RESERVA_POSTERIOR/`. Escribe en `REVISION_2026_11/V4/PRUEBA/`, así que no toca la base de desarrollo. El corte es el menor de los máximos de extracción de la reserva. Solo corre si existe `V4/P4b_decision_v2.json`.

# 0330 · Base analítica v4: correcciones de la evaluación del 04-11-2026 (una fila por guía)

Parte de la versión 3.2 (la que produjo la base usada en el modelado) y aplica los ajustes del director:

| Ajuste | Cambio |
| --- | --- |
| C1 | Lee `LOG_GUIAS_BOGOTA_V4` (0305), que incluye las guías abiertas al extraer. El corte de observación deja de ser el 31-08 23:59: se calcula como el menor de los máximos de `FECHA_EXTRACCION_DW` de guías y movimientos (`CORTE_OBSERVACION = "AUTO"`), o se fija a mano con lo que mostró 0345. Nueva columna `admitida_despues_del_corte`. |
| A1 | Nueva columna `entrega_sin_hora`. Para esas guías la etiqueta de clasificación se conoce al final del día de entrega (o al vencer el plazo, si es antes), no a las 00:00. |
| A2 | T0 = primera salida a zona urbana **posterior** a la admisión (con `TOLERANCIA_T0_MIN` minutos de margen, 0 por defecto). Se guarda también la primera salida sin esa condición (`t0_sin_condicion`) para contar cuántas guías cambian. Las guías con entrega anterior a la admisión quedan con `estado_desenlace = 'ENTREGA_ANTES_DE_ADMISION'` y sin etiqueta. |
| A3 | Nueva columna `estado_guia` (nombre de `PAR.ESTADOSGUIA`). |
| D2 | Nuevas columnas `primera_entrada` (primer proceso 5 o 6 entre la admisión y T0), `h_admision_a_bodega` y `h_bodega_a_t0`. |

Lo demás no cambia: SLA en días hábiles, regla de RED, devoluciones y siniestros aparte, guía seudonimizada con la **misma sal** (los seudónimos coinciden con la v3).

**Salidas.** Carpeta nueva `REVISION_2026_11/V4/`: `GOLD_V4_ENVIO.parquet` y los CSV `P3_*`. La base v3 no se toca.

In [0]:
%pip install "duckdb<2"

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 125.5 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CONFIGURACIÓN · revisar las rutas antes de ejecutar (son las mismas de 0300 y 0330 v2)
import os
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
RUTA_GUIAS      = f"{PATH_DATABRICKS}RESERVA_POSTERIOR/LOG_GUIAS/RAW_V4/LOG_GUIAS_BOGOTA_V4.parquet"   # la crea 0305_GuiasReserva
RUTA_MOVS_GLOB  = f"{PATH_DATABRICKS}RESERVA_POSTERIOR/LOG_SEGUIMIENTO_ENVIO/LOG_SEGUIMIENTO_ENVIO*.parquet"
RUTA_DOCS_GLOB  = f"{PATH_DATABRICKS}RESERVA_POSTERIOR/LOG_DOCUMENTOS/LOG_DOCUMENTOS*.parquet"
PAR = f"{PATH_DATABRICKS}PARAMETRICAS/"
P = {"CIUDADES":   f"{PAR}PAR.CIUDADES.parquet",       "REGIONALES": f"{PAR}PAR.REGIONALES_PAIS.parquet",
     "EST_ENVIO":  f"{PAR}PAR.ESTADOSENVIO.parquet",   "EST_GUIA":   f"{PAR}PAR.ESTADOSGUIA.parquet",
     "RED":        f"{PAR}PAR.RED_OPERATIVA_OFERTA.parquet", "PROCESOS": f"{PAR}PAR.PROCESOS.parquet",
     "ZONAS":      f"{PAR}PAR.ZONASURBANAS.parquet"}
RUTA_CALENDARIO = f"{PAR}DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet"
# Sal para seudonimizar la guía: archivo en el Volume, nunca en el cuaderno ni en GitHub
RUTA_SAL_GUIA   = f"{PATH_DATABRICKS}SEGURIDAD/sal_guia.txt"
# Salidas nuevas de esta versión: carpeta propia, nada de lo existente se sobrescribe
CARPETA_SALIDAS = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/PRUEBA/"   # fase 4: carpeta propia
os.makedirs(CARPETA_SALIDAS, exist_ok=True)
RUTA_GOLD_V3 = f"{CARPETA_SALIDAS}GOLD_V4_ENVIO.parquet"   # se conserva el nombre de la variable

# ESCRITURA EN EL VOLUME (v6, 24-10-2026) · en Databricks serverless el FUSE de /Volumes no acepta las
# escrituras asíncronas de DuckDB ("Async write failed ... errno 5"). El COPY escribe primero en disco
# local y después el archivo se copia completo al Volume.
import os, re, shutil, tempfile, uuid

def copy_a_volume(sql_copy, con=None):
    """Ejecuta un COPY (...) TO '<ruta>' de DuckDB escribiendo en /tmp y copiando el resultado a <ruta>."""
    destino_m = list(re.finditer(r"\bTO\s+'([^']+)'", sql_copy, flags=re.IGNORECASE))[-1]
    destino = destino_m.group(1)
    local = os.path.join(tempfile.gettempdir(), f"duckdb_{uuid.uuid4().hex}_{os.path.basename(destino)}")
    sql_local = sql_copy[:destino_m.start(1)] + local + sql_copy[destino_m.end(1):]
    try:
        (con or duckdb).execute(sql_local)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        try:
            shutil.copyfile(local, destino)
        except OSError:
            dbutils.fs.cp(f"file:{local}", destino)
    finally:
        if os.path.exists(local):
            os.remove(local)
    print(f"Escrito en {destino}")
    return destino

In [0]:
# GUARDA (C2): la reserva solo se abre con la decisión v2 ya registrada
import os, json
RUTA_DECISION = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/P4b_decision_v2.json"
assert os.path.exists(RUTA_DECISION), "No hay P4b_decision_v2.json: la reserva no se abre antes de registrar la decisión"
print("Decisión registrada el", json.load(open(RUTA_DECISION))["fecha_registro"], "· se puede construir la base de la reserva")

Decisión registrada el 2026-11-08T00:46:23-05:00 · se puede construir la base de la reserva


## Paso 0 · Diagnóstico corto (solo conteos)

Antes de construir la base: qué valores trae el calendario, si cubre todos los vencimientos posibles y qué cambia entre las filas repetidas de `RED_OPERATIVA_OFERTA`. Nada de esto muestra guías.

In [0]:
import duckdb
cal = f"read_parquet('{RUTA_CALENDARIO}')"
print("Columnas del calendario:", duckdb.sql(f"DESCRIBE SELECT * FROM {cal}").df()[["column_name", "column_type"]].values.tolist())

# a) Tipos de día en 2026 y si algún día marcado como hábil tiene fecha festiva
d0a = duckdb.sql(f"""
    SELECT TIPO_DIA, ID_TIPO_DIA, COUNT(*) AS dias,
           COUNT(*) FILTER (WHERE FECHA_FESTIVA IS NOT NULL) AS con_fecha_festiva,
           list_distinct(list(dayofweek(CAST("DATE" AS DATE)))) AS dias_semana   -- 0 = domingo
    FROM {cal} WHERE CAST("DATE" AS DATE) BETWEEN DATE '2026-01-01' AND DATE '2026-12-31'
    GROUP BY ALL ORDER BY ALL""").df()
display(d0a)

# b) Cobertura: el calendario debe llegar más allá del último vencimiento posible (envío + plazo máximo)
d0b = duckdb.sql(f"""SELECT MIN(CAST("DATE" AS DATE)) AS primera_fecha, MAX(CAST("DATE" AS DATE)) AS ultima_fecha,
                            COUNT(*) AS filas, COUNT(DISTINCT CAST("DATE" AS DATE)) AS fechas_distintas
                     FROM {cal}""").df()
display(d0b)

# c) RED_OPERATIVA: en las claves repetidas, ¿qué columnas cambian?
red = f"read_parquet('{P['RED']}')"
print("Tipo de ID_ESTADO_RUTA:", duckdb.sql(f"SELECT any_value(typeof(ID_ESTADO_RUTA)) FROM {red}").fetchall()[0][0])
d0c = duckdb.sql(f"""
    WITH g AS (
        SELECT ID_CIUDAD_ORIGEN, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS, COUNT(*) AS n,
               COUNT(DISTINCT TIEMPO_ENTREGA_COMERCIAL_DIA) AS n_dia, COUNT(DISTINCT ID_ESTADO_RUTA) AS n_estado,
               COUNT(DISTINCT ID_IMPEDANCIA) AS n_imp, COUNT(DISTINCT ID_FILTRO_CRITICIDAD) AS n_filtro,
               COUNT(DISTINCT FECHA_MODIFICACION) AS n_fmod,
               COUNT(*) FILTER (WHERE COALESCE(CAST(ID_ESTADO_RUTA AS INT), 0) = 1) AS n_activas
        FROM {red} GROUP BY ALL HAVING COUNT(*) > 1)
    SELECT COUNT(*) AS claves_repetidas,
           AVG((n_dia > 1)::INT)    AS pct_con_sla_distinto,
           AVG((n_estado > 1)::INT) AS pct_estado_distinto,
           AVG((n_imp > 1)::INT)    AS pct_impedancia_distinta,
           AVG((n_filtro > 1)::INT) AS pct_filtro_distinto,
           AVG((n_fmod > 1)::INT)   AS pct_fecha_modificacion_distinta,
           AVG((n_activas = 0)::INT) AS pct_sin_ruta_activa,
           AVG((n_activas > 1)::INT) AS pct_varias_activas
    FROM g""").df()
display(d0c)

# d) Distribución del plazo en días (y si hay plazos no enteros)
d0d = duckdb.sql(f"""SELECT TIEMPO_ENTREGA_COMERCIAL_DIA AS sla_dias, COUNT(*) AS filas_red,
                            AVG((TIEMPO_ENTREGA_COMERCIAL_HOR = TIEMPO_ENTREGA_COMERCIAL_DIA * 24)::INT) AS pct_hor_igual_dia_x24
                     FROM {red} GROUP BY 1 ORDER BY 1""").df()
display(d0d)
for n, t in {"0a_tipos_dia": d0a, "0b_cobertura_calendario": d0b, "0c_red_repetidas": d0c, "0d_sla_dias_red": d0d}.items():
    t.to_csv(f"{CARPETA_SALIDAS}P3_{n}.csv", index=False)

Columnas del calendario: [['TIME_ID', 'BIGINT'], ['DATE', 'TIMESTAMP'], ['DAY', 'VARCHAR'], ['MONTH', 'VARCHAR'], ['YEAR', 'BIGINT'], ['DAY_OF_MONTH', 'BIGINT'], ['WEEK_OF_YEAR', 'BIGINT'], ['MONTH_OF_YEAR', 'BIGINT'], ['QUARTER_OF_YEAR', 'BIGINT'], ['SEMESTER_OF_YEAR', 'BIGINT'], ['SEMESTER', 'VARCHAR'], ['FECHA_FESTIVA', 'TIMESTAMP'], ['TIPO_DIA', 'VARCHAR'], ['ID_TIPO_DIA', 'BIGINT']]


TIPO_DIA,ID_TIPO_DIA,dias,con_fecha_festiva,dias_semana
DIA DOMINICAL,2,52,0,List(0)
DIA FESTIVO,1,18,18,"List(5, 2, 1, 4)"
DIA HABIL,0,295,0,"List(4, 3, 2, 1, 6, 5)"


primera_fecha,ultima_fecha,filas,fechas_distintas
1900-01-01T00:00:00.000Z,2032-12-31T00:00:00.000Z,17726,17726


Tipo de ID_ESTADO_RUTA: BOOLEAN


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

claves_repetidas,pct_con_sla_distinto,pct_estado_distinto,pct_impedancia_distinta,pct_filtro_distinto,pct_fecha_modificacion_distinta,pct_sin_ruta_activa,pct_varias_activas
1327822,0.29762648909266454,4.3981798765195936E-4,0.0,0.0,0.45473263735651315,0.5737282557451224,0.4258319262672256


sla_dias,filas_red,pct_hor_igual_dia_x24
0.0,7785,1.0
6.94E-4,2829,0.0
0.001388,1,0.0
0.002083,9,0.0
0.003472,19,0.0
0.004861,6,0.0
0.005555,16,0.0
0.00625,4,0.0
0.006944,123,0.0
0.007638,57,0.0


In [0]:
# (La versión 3.2 repetía aquí el Paso 0; se dejó una sola vez.)

## Parámetros

Lo que sí está decidido (21-10-2026): días hábiles, sin esperar la confirmación de Servienvios, y alternativa A. Lo demás queda como parámetro, con su sensibilidad al final. La variante principal se fija **antes** de ver la prevalencia y no se cambia por acercarse al 20 % que reporta la empresa.

`CONDICION_DIA_HABIL` es una expresión SQL sobre las columnas del calendario. El valor propuesto toma como hábil lo que el calendario marca `DIA HABIL` (incluye el sábado) y que no tenga fecha festiva. Si el Paso 0 muestra otros textos en `TIPO_DIA`, se ajusta aquí.

In [0]:
# Momento de la extracción (C1): máximos de FECHA_EXTRACCION_DW en guías y movimientos
def _max(ruta, col):
    cols = set(duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{ruta}')").df().column_name)
    col = col if col in cols else "FECHA_HORA_MOVIMIENTO"
    return pd.Timestamp(duckdb.sql(f"SELECT MAX({col}) FROM read_parquet('{ruta}')").fetchone()[0]), col
import pandas as pd
MAX_EXTRACCION_GUIAS, _ = _max(RUTA_GUIAS, "FECHA_EXTRACCION_DW")
MAX_EXTRACCION_MOVS, COL_MOVS = _max(RUTA_MOVS_GLOB, "FECHA_EXTRACCION_DW")
CORTE_AUTO = min(MAX_EXTRACCION_GUIAS, MAX_EXTRACCION_MOVS)
print(f"Extracción guías: {MAX_EXTRACCION_GUIAS} · movimientos ({COL_MOVS}): {MAX_EXTRACCION_MOVS} · corte automático: {CORTE_AUTO}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extracción guías: 2026-11-05 09:45:09 · movimientos (FECHA_EXTRACCION_DW): 2026-11-05 10:20:21 · corte automático: 2026-11-05 09:45:09


In [0]:
CAMPO_ENTREGA       = "FECHA_ENTREGA"        # columna de LOG_GUIAS validada en 0310 como entrega efectiva
CORTE_OBSERVACION   = "AUTO"                 # C1: "AUTO" = menor de los máximos de extracción (celda anterior), o una fecha fija
CORTE_OBSERVACION   = str(CORTE_AUTO) if CORTE_OBSERVACION == "AUTO" else CORTE_OBSERVACION
assert pd.Timestamp(CORTE_OBSERVACION) >= pd.Timestamp("2026-10-01"), f"Corte de la reserva inesperado: {CORTE_OBSERVACION}"
print("Corte de observación de la reserva:", CORTE_OBSERVACION)
TOLERANCIA_T0_MIN   = 0                      # A2: minutos de desfase de registro aceptados entre admisión y T0 (fijar con 0345)
ID_SALIDA_ZONA      = 9
SLA_EN_DIAS_HABILES = True                   # decisión 21-10; False reproduce la regla v2 (horas continuas) para comparar
CUENTA_DIA_ADMISION = False                  # False: el plazo empieza el día hábil siguiente a la admisión (principal)
REGLA_RED           = "ACTIVA_MAS_RECIENTE"  # principal; sensibilidad: "MAXIMO" (el plazo más holgado de las rutas activas)
COL_FECHA_CALENDARIO = "DATE"
CONDICION_DIA_HABIL = "UPPER(TRIM(TIPO_DIA)) = 'DIA HABIL' AND FECHA_FESTIVA IS NULL"   # ajustar con el Paso 0
ESTADOS_DEVOLUCION  = ["ENTREGADO A REMITENTE"]
ESTADOS_SINIESTRO   = ["SINIESTRADO"]
INCUMPLE_SI_IGUAL   = False                  # solo aplica con SLA_EN_DIAS_HABILES = False

Corte de observación de la reserva: 2026-11-05 09:45:09


## Funciones (no es necesario modificarlas)

In [0]:
# =====================================================================================
# BASE ANALÍTICA V3: UNA FILA POR GUÍA, T0 EXPLÍCITO, SLA EN DÍAS HÁBILES CON REGLA DE RED
# EXPLÍCITA, DEVOLUCIONES Y SINIESTROS APARTE Y GUÍA SEUDONIMIZADA.
# Parte de construir_gold_v2 (0330 v2). Cambios marcados con «v3».
# =====================================================================================
import duckdb, secrets

def leer_o_crear_sal(ruta):
    """La sal vive en el Volume (no en el cuaderno ni en GitHub). Si no existe, se crea una vez."""
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        with open(ruta, "w") as f:
            f.write(secrets.token_hex(16))
        print(f"Sal nueva creada en {ruta}. No copiarla al cuaderno ni subirla al repositorio.")
    with open(ruta) as f:
        sal = f.read().strip()
    assert len(sal) >= 16 and "'" not in sal, "La sal debe tener al menos 16 caracteres y no llevar comillas"
    return sal

def _lista_sql(valores):
    return ", ".join("'" + v.upper().replace("'", "''") + "'" for v in valores)

def sql_calendario(ruta_calendario, col_fecha, condicion_habil):
    """CTE del calendario hábil: cada día hábil recibe un número consecutivo (n_hab)."""
    return f"""
    CAL AS (
        SELECT CAST("{col_fecha}" AS DATE) AS fecha, bool_and(COALESCE({condicion_habil}, FALSE)) AS es_habil
        FROM read_parquet('{ruta_calendario}') GROUP BY 1
    ),
    HAB AS (SELECT fecha, es_habil, SUM(es_habil::INT) OVER (ORDER BY fecha) AS n_hab FROM CAL),
    HABIL AS (SELECT fecha, n_hab FROM HAB WHERE es_habil)"""

def sql_limite_habil(alias_h0, alias_hv, col_envio, col_dias, cuenta_dia_admision):
    """JOIN que ubica el día hábil del vencimiento. El límite es el inicio del día siguiente (fin del día hábil)."""
    ajuste = f"CASE WHEN {alias_h0}.es_habil THEN 1 ELSE 0 END" if cuenta_dia_admision else "0"
    return f"""LEFT JOIN HAB {alias_h0} ON {alias_h0}.fecha = CAST({col_envio} AS DATE)
        LEFT JOIN HABIL {alias_hv} ON {alias_hv}.n_hab = {alias_h0}.n_hab + {col_dias} - {ajuste}"""

def construir_gold_v3(R, P, RUTA_SALIDA, RUTA_CALENDARIO, SAL_GUIA, CAMPO_ENTREGA="FECHA_ENTREGA", ID_SALIDA_ZONA=9,
                      TOLERANCIA_T0_MIN=0,
                      CORTE_OBSERVACION="2026-08-31 23:59:59", SLA_EN_DIAS_HABILES=True, CUENTA_DIA_ADMISION=False,
                      REGLA_RED="ACTIVA_MAS_RECIENTE", COL_FECHA_CALENDARIO="DATE",
                      CONDICION_DIA_HABIL="UPPER(TRIM(TIPO_DIA)) = 'DIA HABIL' AND FECHA_FESTIVA IS NULL",
                      ESTADOS_DEVOLUCION=("ENTREGADO A REMITENTE",), ESTADOS_SINIESTRO=("SINIESTRADO",),
                      INCUMPLE_SI_IGUAL=False):
    assert REGLA_RED in ("ACTIVA_MAS_RECIENTE", "MAXIMO"), "REGLA_RED debe ser ACTIVA_MAS_RECIENTE o MAXIMO"
    # Con días hábiles el límite es el inicio del día siguiente: entregar en ese instante o después incumple
    op = ">=" if (SLA_EN_DIAS_HABILES or INCUMPLE_SI_IGUAL) else ">"
    col_sla = "sla_dias_reciente" if REGLA_RED == "ACTIVA_MAS_RECIENTE" else "sla_dias_max"
    limite = ("epoch_ms(CAST(HV.fecha AS TIMESTAMP) + INTERVAL 1 DAY)" if SLA_EN_DIAS_HABILES
              else "epoch_ms(B.FECHA_ENVIO) + CAST(B.sla_dias * 24 * 3600000 AS BIGINT)")
    con = duckdb.connect()
    copy_a_volume(f"""
    COPY (
    WITH
    GUI AS (
        SELECT CAST(NUMERO_GUIA AS BIGINT) AS guia, ID_TIPOGUIA, ID_CLIENTE, ID_PAIS_ORIGEN, ID_CIUDAD_ORIGEN,
               ID_PAIS_DESTINO, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS, ID_PRODUCTO, ID_SUBPRODUCTO,
               ID_ESTADOGUIA, ID_ESTADOENVIO, FECHA_ENVIO, {CAMPO_ENTREGA} AS fecha_entrega,
               FECHA_EXTRACCION_DW AS fecha_extraccion_guia,
               NUMERO_PIEZAS AS piezas_declaradas, PESO_GUIA AS peso_declarado_kg,
               LARGO_ENVIO, ANCHO_ENVIO, ALTO_ENVIO
        FROM read_parquet('{R["GUIAS"]}')
    ),
    -- v4 (A2): movimientos de la población con la admisión de su guía
    MOV AS (
        SELECT CAST(M.NUMERO_GUIA AS BIGINT) AS guia, M.ID_PROCESO, M.NUMERO_MOVIMI, M.CONSECUT_MOVIMI,
               M.FECHA_HORA_MOVIMIENTO, G.FECHA_ENVIO
        FROM read_parquet('{R["MOVS_GLOB"]}') M
        JOIN GUI G ON G.guia = CAST(M.NUMERO_GUIA AS BIGINT)
    ),
    -- v4 (A2): T0 = PRIMERA salida a zona urbana POSTERIOR a la admisión (desempate determinista)
    SAL AS (
        SELECT guia, ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI, FECHA_HORA_MOVIMIENTO AS t0,
               COUNT(*) OVER (PARTITION BY guia) AS n_salidas_zona
        FROM MOV
        WHERE ID_PROCESO = {ID_SALIDA_ZONA}
          AND FECHA_HORA_MOVIMIENTO >= FECHA_ENVIO - INTERVAL ({int(TOLERANCIA_T0_MIN)}) MINUTE
        QUALIFY ROW_NUMBER() OVER (PARTITION BY guia ORDER BY FECHA_HORA_MOVIMIENTO, NUMERO_MOVIMI, CONSECUT_MOVIMI) = 1
    ),
    -- v4: primera salida sin condición (la regla anterior), solo para contar cuántas guías cambian de T0
    SAL0 AS (SELECT guia, MIN(FECHA_HORA_MOVIMIENTO) AS t0_sin_condicion FROM MOV WHERE ID_PROCESO = {ID_SALIDA_ZONA} GROUP BY 1),
    -- v4 (D2): primera entrada a bodega (proceso 5 o 6) entre la admisión y T0
    ENT AS (
        SELECT MV.guia, MIN(MV.FECHA_HORA_MOVIMIENTO) AS primera_entrada
        FROM MOV MV JOIN SAL S USING (guia)
        WHERE MV.ID_PROCESO IN (5, 6) AND MV.FECHA_HORA_MOVIMIENTO >= MV.FECHA_ENVIO AND MV.FECHA_HORA_MOVIMIENTO <= S.t0
        GROUP BY 1
    ),
    -- v4 (A3): nombre del estado de la guía
    EGU AS (
        SELECT ID_ESTADOGUIA,
               CASE WHEN COUNT(DISTINCT NOMBRE_ESTADOGUIA) = 1 THEN UPPER(TRIM(MIN(NOMBRE_ESTADOGUIA))) END AS estado_guia
        FROM read_parquet('{P["EST_GUIA"]}') GROUP BY ALL
    ),
    -- Documento del movimiento T0, deduplicado por su clave (el conteo previo va en el control)
    DOC AS (
        SELECT ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI, ID_TIPO_ZONA_URBA, ID_CIUDAD_ZONA_URBA, ID_ZONA_URBA
        FROM read_parquet('{R["DOCS_GLOB"]}')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI
                                   ORDER BY FECHA_EXTRACCION_DW DESC, FECHA_MODIFICACION DESC) = 1
    ),
    -- v3: RED_OPERATIVA reducida a UNA fila por clave con regla explícita.
    --     Se prefieren las rutas activas; entre ellas, la de FECHA_MODIFICACION más reciente.
    REDR AS (
        SELECT ID_CIUDAD_ORIGEN, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS,
               TIEMPO_ENTREGA_COMERCIAL_DIA AS dia, TIEMPO_ENTREGA_COMERCIAL_HOR AS hor, FECHA_MODIFICACION,
               COALESCE(CAST(ID_ESTADO_RUTA AS INT), 0) AS activa,
               MAX(COALESCE(CAST(ID_ESTADO_RUTA AS INT), 0))
                   OVER (PARTITION BY ID_CIUDAD_ORIGEN, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS) AS hay_activa
        FROM read_parquet('{P["RED"]}')
    ),
    RED AS (
        SELECT ID_CIUDAD_ORIGEN, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS,
               COUNT(*) AS n_red,
               COUNT(DISTINCT dia) AS n_valores_sla,
               MAX(hay_activa) = 0 AS red_sin_ruta_activa,
               first(dia ORDER BY activa DESC, FECHA_MODIFICACION DESC NULLS LAST, dia DESC) AS dia_reciente,
               first(hor ORDER BY activa DESC, FECHA_MODIFICACION DESC NULLS LAST, dia DESC) AS hor_reciente,
               MAX(dia) FILTER (WHERE activa = hay_activa) AS dia_max
        FROM REDR GROUP BY ALL
    ),
    ZON AS (
        SELECT ID_CIUDAD, ID_TIPO_ZONA_URBA, ID_ZONA_URBA, COUNT(*) AS n_zona,
               CASE WHEN COUNT(DISTINCT NOMBRE_ZONA_URBA) = 1 THEN MIN(NOMBRE_ZONA_URBA) END AS zona_urbana,
               CASE WHEN COUNT(DISTINCT CATEGORIA_ZONA_URBA) = 1 THEN MIN(CATEGORIA_ZONA_URBA) END AS categoria_zona
        FROM read_parquet('{P["ZONAS"]}') WHERE ID_PAIS = 1 GROUP BY ALL
    ),
    CIU AS (
        SELECT ID_PAIS, ID_CIUDAD, COUNT(*) AS n_ciu,
               CASE WHEN COUNT(DISTINCT NOMBRE_CIUDAD) = 1 THEN MIN(NOMBRE_CIUDAD) END AS ciudad,
               CASE WHEN COUNT(DISTINCT ID_REGIONAL)   = 1 THEN MIN(ID_REGIONAL)   END AS id_regional
        FROM read_parquet('{P["CIUDADES"]}') GROUP BY ALL
    ),
    REG AS (
        SELECT ID_REGIONAL, CASE WHEN COUNT(DISTINCT NOMBRE_REGIONAL) = 1 THEN MIN(NOMBRE_REGIONAL) END AS regional
        FROM read_parquet('{P["REGIONALES"]}') GROUP BY ALL
    ),
    -- v3: estado del envío para separar devoluciones y siniestros
    EEN AS (
        SELECT ID_ESTADOENVIO,
               CASE WHEN COUNT(DISTINCT NOMBRE_ESTADOENVIO) = 1 THEN UPPER(TRIM(MIN(NOMBRE_ESTADOENVIO))) END AS estado_envio
        FROM read_parquet('{P["EST_ENVIO"]}') GROUP BY ALL
    ),
    {sql_calendario(RUTA_CALENDARIO, COL_FECHA_CALENDARIO, CONDICION_DIA_HABIL)},
    BASE AS (
        SELECT G.*, S.t0, S.n_salidas_zona, S0.t0_sin_condicion, EN.primera_entrada, EG.estado_guia,
               D.ID_TIPO_ZONA_URBA, D.ID_ZONA_URBA,
               Z.zona_urbana, Z.categoria_zona, Z.n_zona,
               CO.ciudad AS ciudad_origen, RO.regional AS regional_origen,
               E.estado_envio,
               R2.n_red, R2.n_valores_sla, R2.red_sin_ruta_activa,
               CAST(CEIL(R2.dia_reciente) AS INT) AS sla_dias_reciente,
               CAST(CEIL(R2.dia_max) AS INT)      AS sla_dias_max,
               R2.hor_reciente                    AS sla_h_continuas,      -- valor en horas de la fila elegida (comparación v2)
               (COALESCE(G.ID_TIEMPO,0) = 0 OR COALESCE(G.ID_MEDTRANS,0) = 0) AS sla_con_sustitucion_1
        FROM GUI G
        JOIN SAL S USING (guia)                                   -- población: guías con salida a zona posterior a la admisión
        LEFT JOIN SAL0 S0 USING (guia)
        LEFT JOIN ENT EN USING (guia)
        LEFT JOIN EGU EG ON EG.ID_ESTADOGUIA = G.ID_ESTADOGUIA
        LEFT JOIN DOC D ON D.ID_PROCESO = S.ID_PROCESO AND D.NUMERO_MOVIMI = S.NUMERO_MOVIMI
                       AND D.CONSECUT_MOVIMI = S.CONSECUT_MOVIMI
        LEFT JOIN ZON Z ON Z.ID_CIUDAD = D.ID_CIUDAD_ZONA_URBA AND Z.ID_TIPO_ZONA_URBA = D.ID_TIPO_ZONA_URBA
                       AND Z.ID_ZONA_URBA = D.ID_ZONA_URBA
        LEFT JOIN CIU CO ON CO.ID_PAIS = G.ID_PAIS_ORIGEN AND CO.ID_CIUDAD = G.ID_CIUDAD_ORIGEN
        LEFT JOIN REG RO ON RO.ID_REGIONAL = CO.id_regional
        LEFT JOIN EEN E  ON E.ID_ESTADOENVIO = G.ID_ESTADOENVIO
        LEFT JOIN RED R2 ON R2.ID_CIUDAD_ORIGEN = G.ID_CIUDAD_ORIGEN AND R2.ID_CIUDAD_DESTINO = G.ID_CIUDAD_DESTINO
                        AND R2.ID_TIEMPO   = CASE WHEN COALESCE(G.ID_TIEMPO,0)   <> 0 THEN G.ID_TIEMPO   ELSE 1 END
                        AND R2.ID_MEDTRANS = CASE WHEN COALESCE(G.ID_MEDTRANS,0) <> 0 THEN G.ID_MEDTRANS ELSE 1 END
    ),
    BASE2 AS (SELECT *, {col_sla} AS sla_dias FROM BASE),
    -- v3: vencimiento al final del n-ésimo día hábil (o en horas continuas si SLA_EN_DIAS_HABILES = False)
    VEN AS (
        SELECT B.*, {limite} AS limite_ms
        FROM BASE2 B
        {sql_limite_habil("H0", "HV", "B.FECHA_ENVIO", "B.sla_dias", CUENTA_DIA_ADMISION)}
    ),
    ETI AS (
        SELECT *,
            estado_envio IN ({_lista_sql(ESTADOS_DEVOLUCION)}) AS es_devolucion,
            estado_envio IN ({_lista_sql(ESTADOS_SINIESTRO)})  AS es_siniestro,
            -- v4 (A1): entrega registrada sin hora (00:00:00): la fecha sirve para la etiqueta, la hora no
            COALESCE(fecha_entrega = date_trunc('day', fecha_entrega), FALSE) AS entrega_sin_hora,
            -- v4 (A2): entrega anterior a la admisión (si no tiene hora, se compara por fecha)
            COALESCE(CASE WHEN fecha_entrega = date_trunc('day', fecha_entrega)
                          THEN CAST(fecha_entrega AS DATE) < CAST(FECHA_ENVIO AS DATE)
                          ELSE fecha_entrega < FECHA_ENVIO END, FALSE)  AS entrega_antes_de_admision,
            (epoch_ms(t0)            - epoch_ms(FECHA_ENVIO)) / 3600000.0 AS h_admision_a_t0,
            (epoch_ms(fecha_entrega) - epoch_ms(t0))          / 3600000.0 AS h_ultima_milla_bruta,
            (epoch_ms(fecha_entrega) - epoch_ms(FECHA_ENVIO)) / 3600000.0 AS h_total,
            epoch_ms(TIMESTAMP '{CORTE_OBSERVACION}')                       AS corte_ms
        FROM VEN
    ),
    FIN AS (
        SELECT *,
            COALESCE(es_devolucion OR es_siniestro OR entrega_antes_de_admision, FALSE) AS fuera_de_desenlace,
            -- v4 (A1): instante en que se conoce la entrega; sin hora = final de ese día
            CASE WHEN entrega_sin_hora THEN epoch_ms(fecha_entrega) + 86399000 ELSE epoch_ms(fecha_entrega) END AS entrega_conocida_ms,
            sla_dias IS NOT NULL AND limite_ms IS NULL     AS vencimiento_sin_calendario
        FROM ETI
    )
    SELECT md5(CAST(guia AS VARCHAR) || '{SAL_GUIA}') AS guia,                 -- v3: seudónimo, nunca la guía real
        * EXCLUDE (guia, corte_ms, limite_ms, es_devolucion, es_siniestro, sla_con_sustitucion_1, fuera_de_desenlace, entrega_conocida_ms),
        (epoch_ms(primera_entrada) - epoch_ms(FECHA_ENVIO)) / 3600000.0     AS h_admision_a_bodega,   -- v4 (D2)
        (epoch_ms(t0) - epoch_ms(primera_entrada)) / 3600000.0              AS h_bodega_a_t0,         -- v4 (D2)
        FECHA_ENVIO > TIMESTAMP '{CORTE_OBSERVACION}'                         AS admitida_despues_del_corte,
        CAST(sla_dias * 24 AS DOUBLE)                                       AS sla_h,   -- plazo nominal en horas (variable)
        make_timestamp((limite_ms - 1000) * 1000)                           AS vencimiento_sla,   -- 23:59:59 del último día
        (limite_ms - epoch_ms(t0)) / 3600000.0                              AS margen_sla_en_t0_h,  -- predictor válido en T0
        CASE WHEN fuera_de_desenlace THEN NULL ELSE h_ultima_milla_bruta END AS h_ultima_milla,
        CASE WHEN sla_dias IS NULL THEN 'SIN_SLA'
             WHEN sla_con_sustitucion_1 THEN 'SLA_CON_SUSTITUCION'
             WHEN n_valores_sla > 1 THEN 'SLA_POR_REGLA'
             ELSE 'SLA_UNICO' END                                           AS origen_sla,
        CASE WHEN es_devolucion THEN 'DEVUELTO'
             WHEN es_siniestro  THEN 'SINIESTRADO'
             WHEN entrega_antes_de_admision THEN 'ENTREGA_ANTES_DE_ADMISION'
             WHEN fecha_entrega IS NOT NULL                     THEN 'ENTREGADO'
             WHEN limite_ms IS NOT NULL AND limite_ms <= corte_ms THEN 'ABIERTO_VENCIDO'
             ELSE 'ABIERTO_NO_VENCIDO' END                                  AS estado_desenlace,
        -- Etiqueta: 1/0 cuando es identificable; NULL si no (abierto no vencido, sin SLA, sin calendario,
        -- devolución o siniestro)
        CASE WHEN fuera_de_desenlace OR limite_ms IS NULL       THEN NULL
             WHEN fecha_entrega IS NOT NULL THEN (epoch_ms(fecha_entrega) {op} limite_ms)::INT
             WHEN limite_ms <= corte_ms                         THEN 1
             ELSE NULL END                                                  AS incumplimiento_sla,
        -- Momento en que cada etiqueta quedó conocida (para cortes y agregados sin futuro)
        CASE WHEN NOT fuera_de_desenlace
             THEN make_timestamp(LEAST(entrega_conocida_ms, limite_ms) * 1000) END AS t_etiqueta_clf,   -- v4 (A1)
        CASE WHEN NOT fuera_de_desenlace THEN fecha_entrega END             AS t_etiqueta_reg,
        (epoch_ms(t0) >= limite_ms)                                         AS vencido_antes_de_t0
    FROM FIN
    ) TO '{RUTA_SALIDA}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """, con)
    con.close()

def controles_gold_v3(RUTA, CORTE_OBSERVACION):
    """Pruebas de propiedades. Falla con mensaje claro; imprime solo agregados."""
    q = lambda s: duckdb.sql(s.replace("{T}", f"read_parquet('{RUTA}')")).df()
    principal = "origen_sla IN ('SLA_UNICO', 'SLA_POR_REGLA')"
    c = q(f"""SELECT COUNT(*) filas, COUNT(DISTINCT guia) guias,
                    COUNT(*) FILTER (WHERE incumplimiento_sla NOT IN (0,1)) etiqueta_fuera_dominio,
                    COUNT(*) FILTER (WHERE incumplimiento_sla IS NULL) etiqueta_nula,
                    COUNT(*) FILTER (WHERE h_ultima_milla < 0) ultima_milla_negativa,
                    COUNT(*) FILTER (WHERE h_admision_a_t0 < 0) t0_antes_de_admision,
                    COUNT(*) FILTER (WHERE t0_sin_condicion < t0) cambian_de_t0_por_a2,
                    COUNT(*) FILTER (WHERE estado_desenlace = 'ENTREGA_ANTES_DE_ADMISION') entrega_antes_de_admision,
                    COUNT(*) FILTER (WHERE entrega_sin_hora) entrega_sin_hora,
                    COUNT(*) FILTER (WHERE estado_desenlace = 'ABIERTO_VENCIDO') abiertas_vencidas,
                    COUNT(*) FILTER (WHERE estado_desenlace = 'ABIERTO_NO_VENCIDO') abiertas_no_vencidas,
                    COUNT(*) FILTER (WHERE admitida_despues_del_corte) admitidas_despues_del_corte,
                    COUNT(*) FILTER (WHERE vencimiento_sin_calendario) sin_vencimiento_por_calendario,
                    COUNT(*) FILTER (WHERE vencimiento_sla < FECHA_ENVIO) vencimiento_antes_de_admision,
                    COUNT(*) FILTER (WHERE vencido_antes_de_t0) vencidas_antes_de_t0,
                    COUNT(*) FILTER (WHERE estado_desenlace = 'DEVUELTO') devueltas,
                    COUNT(*) FILTER (WHERE estado_desenlace = 'SINIESTRADO') siniestradas,
                    COUNT(*) FILTER (WHERE {principal}) con_sla_principal,
                    AVG(incumplimiento_sla) FILTER (WHERE {principal}) prevalencia_principal
             FROM {{T}}""")
    assert c.loc[0, "filas"] == c.loc[0, "guias"], "Hay más filas que guías: fan-out sin resolver"
    assert c.loc[0, "etiqueta_fuera_dominio"] == 0, "incumplimiento_sla fuera de {0,1}"
    tablas = {
        "control": c,
        "etiqueta_por_estado": q("""SELECT estado_desenlace, origen_sla, COUNT(*) guias,
                                      AVG(incumplimiento_sla) prevalencia, COUNT(incumplimiento_sla) n_etiquetadas
                               FROM {T} GROUP BY ALL ORDER BY guias DESC"""),
        "por_mes": q(f"""SELECT date_trunc('month', FECHA_ENVIO) mes, COUNT(*) guias,
                          COUNT(*) FILTER (WHERE estado_desenlace = 'ABIERTO_VENCIDO') abiertas_vencidas,
                          COUNT(*) FILTER (WHERE estado_desenlace = 'ABIERTO_NO_VENCIDO') abiertas_no_vencidas,
                          AVG((estado_desenlace='ENTREGADO')::INT) pct_entregado,
                          AVG((vencido_antes_de_t0)::INT) pct_vencido_antes_de_t0,
                          AVG(incumplimiento_sla) FILTER (WHERE {principal}) prevalencia_principal,
                          AVG(incumplimiento_sla) FILTER (WHERE origen_sla = 'SLA_UNICO') prevalencia_solo_sla_unico
                        FROM {{T}} GROUP BY 1 ORDER BY 1"""),
        "sla_dias": q(f"""SELECT sla_dias, COUNT(*) guias, AVG(incumplimiento_sla) prevalencia,
                           median(h_admision_a_t0) mediana_h_admision_a_t0, median(margen_sla_en_t0_h) mediana_margen_h
                         FROM {{T}} WHERE {principal} GROUP BY 1 ORDER BY 1"""),
    }
    return tablas

def sensibilidad_sla(RUTA, RUTA_CALENDARIO, COL_FECHA_CALENDARIO, CONDICION_DIA_HABIL, CORTE_OBSERVACION):
    """Prevalencia y % vencido antes de T0 en las variantes de la regla del SLA, sobre la misma población
    (SLA único o por regla, sin devoluciones ni siniestros). La variante 1 es la principal."""
    variantes = [(1, False, "sla_dias_reciente"), (2, True, "sla_dias_reciente"),
                 (3, False, "sla_dias_max"),      (4, True, "sla_dias_max")]
    partes = []
    for k, cuenta, col in variantes:
        partes.append(f"""
        SELECT {k} AS variante, 'días hábiles' AS unidad, {cuenta} AS cuenta_dia_admision,
               '{'ACTIVA_MAS_RECIENTE' if col == 'sla_dias_reciente' else 'MAXIMO'}' AS regla_red,
               epoch_ms(CAST(HV.fecha AS TIMESTAMP) + INTERVAL 1 DAY) AS lim, B.*
        FROM POB B {sql_limite_habil("H0", "HV", "B.FECHA_ENVIO", "B." + col, cuenta)}""")
    partes.append("""
        SELECT 0, 'horas continuas (regla v2)', FALSE, 'ACTIVA_MAS_RECIENTE',
               epoch_ms(B.FECHA_ENVIO) + CAST(B.sla_dias_reciente * 24 * 3600000 AS BIGINT), B.*
        FROM POB B""")
    sql = f"""
    WITH {sql_calendario(RUTA_CALENDARIO, COL_FECHA_CALENDARIO, CONDICION_DIA_HABIL).strip()},
    POB AS (SELECT FECHA_ENVIO, fecha_entrega, t0, sla_dias_reciente, sla_dias_max
            FROM read_parquet('{RUTA}')
            WHERE origen_sla IN ('SLA_UNICO', 'SLA_POR_REGLA') AND estado_desenlace NOT IN ('DEVUELTO', 'SINIESTRADO')),
    V AS ({" UNION ALL ".join(partes)})
    SELECT variante, unidad, cuenta_dia_admision, regla_red, COUNT(*) AS guias,
           AVG(CASE WHEN lim IS NULL THEN NULL
                    WHEN fecha_entrega IS NOT NULL THEN (epoch_ms(fecha_entrega) >= lim)::INT
                    WHEN lim <= epoch_ms(TIMESTAMP '{CORTE_OBSERVACION}') THEN 1 END) AS prevalencia,
           AVG((epoch_ms(t0) >= lim)::INT) AS pct_vencido_antes_de_t0,
           COUNT(*) FILTER (WHERE lim IS NULL) AS sin_vencimiento
    FROM V GROUP BY ALL ORDER BY variante"""
    return duckdb.sql(sql).df()

## Construcción y controles

In [0]:
SAL_GUIA = leer_o_crear_sal(RUTA_SAL_GUIA)
R = {"GUIAS": RUTA_GUIAS, "MOVS_GLOB": RUTA_MOVS_GLOB, "DOCS_GLOB": RUTA_DOCS_GLOB}
construir_gold_v3(R, P, RUTA_GOLD_V3, RUTA_CALENDARIO, SAL_GUIA, CAMPO_ENTREGA=CAMPO_ENTREGA,
                  ID_SALIDA_ZONA=ID_SALIDA_ZONA, TOLERANCIA_T0_MIN=TOLERANCIA_T0_MIN, CORTE_OBSERVACION=CORTE_OBSERVACION,
                  SLA_EN_DIAS_HABILES=SLA_EN_DIAS_HABILES, CUENTA_DIA_ADMISION=CUENTA_DIA_ADMISION,
                  REGLA_RED=REGLA_RED, COL_FECHA_CALENDARIO=COL_FECHA_CALENDARIO,
                  CONDICION_DIA_HABIL=CONDICION_DIA_HABIL, ESTADOS_DEVOLUCION=ESTADOS_DEVOLUCION,
                  ESTADOS_SINIESTRO=ESTADOS_SINIESTRO, INCUMPLE_SI_IGUAL=INCUMPLE_SI_IGUAL)
del SAL_GUIA
tablas = controles_gold_v3(RUTA_GOLD_V3, CORTE_OBSERVACION)   # se detiene con error si falla la unidad o el dominio
for nombre, t in tablas.items():
    print(nombre); display(t); t.to_csv(f"{CARPETA_SALIDAS}P3_{nombre}.csv", index=False)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Escrito en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_11/V4/PRUEBA/GOLD_V4_ENVIO.parquet
control


filas,guias,etiqueta_fuera_dominio,etiqueta_nula,ultima_milla_negativa,t0_antes_de_admision,cambian_de_t0_por_a2,entrega_antes_de_admision,entrega_sin_hora,abiertas_vencidas,abiertas_no_vencidas,admitidas_despues_del_corte,sin_vencimiento_por_calendario,vencimiento_antes_de_admision,vencidas_antes_de_t0,devueltas,siniestradas,con_sla_principal,prevalencia_principal
749884,749884,0,23191,3135,0,149,11,7906,7714,7059,1,0,0,303306,16003,109,742277,0.45251559777689093


etiqueta_por_estado


estado_desenlace,origen_sla,guias,prevalencia,n_etiquetadas
ENTREGADO,SLA_UNICO,708323,0.44476601776308267,708323
DEVUELTO,SLA_UNICO,15978,null,0
ABIERTO_VENCIDO,SLA_UNICO,7627,1.0,7627
ENTREGADO,SLA_CON_SUSTITUCION,7479,0.3848108035833668,7479
ABIERTO_NO_VENCIDO,SLA_UNICO,6982,null,0
ENTREGADO,SLA_POR_REGLA,3177,0.8580421781554926,3177
SINIESTRADO,SLA_UNICO,106,null,0
ABIERTO_NO_VENCIDO,SLA_CON_SUSTITUCION,73,null,0
ABIERTO_VENCIDO,SLA_POR_REGLA,46,1.0,46
ABIERTO_VENCIDO,SLA_CON_SUSTITUCION,41,1.0,41


por_mes


mes,guias,abiertas_vencidas,abiertas_no_vencidas,pct_entregado,pct_vencido_antes_de_t0,prevalencia_principal,prevalencia_solo_sla_unico
2026-09-01T00:00:00.000Z,370700,429,0,0.9736471540329107,0.4113145144756294,0.4447087698896356,0.4425210518332931
2026-10-01T00:00:00.000Z,353001,2870,29,0.9725836470718213,0.4137887076167355,0.46394406900880125,0.4624044609796213
2026-11-01T00:00:00.000Z,26183,4415,7030,0.562731543367834,0.18210289118893938,0.393084092353315,0.39260871873838477


sla_dias


sla_dias,guias,prevalencia,mediana_h_admision_a_t0,mediana_margen_h
0,1131,0.3197353914002205,3.2952777777777778,12.669166666666667
1,619200,0.450180769346326,24.538333333333334,13.9475
2,108966,0.49027054466149017,65.97212916666666,13.915555555555555
3,7251,0.2803083391730904,66.80994527777777,37.691944444444445
4,3201,0.24459287531806614,88.76199916666667,38.63388888888889
5,1905,0.2145512480084971,94.68120361111112,62.06166666666667
6,370,0.1307901907356948,94.19392916666666,87.44680555555556
7,126,0.3412698412698413,138.70083791666667,63.776944444444446
8,115,0.19130434782608696,87.4375,159.26583333333335
9,2,0.5,319.82972222222224,-45.22972222222222


## Sensibilidad de la regla del SLA

Misma población en todas las filas (SLA único o por regla, sin devoluciones ni siniestros). La variante 1 es la principal y se fijó antes de ver estos números; las demás se reportan tal cual. La fila 0 reproduce la regla de la v2 (horas continuas desde la admisión) con la misma fila de RED, para ver cuánto explica el cambio de unidad.

In [0]:
sens = sensibilidad_sla(RUTA_GOLD_V3, RUTA_CALENDARIO, COL_FECHA_CALENDARIO, CONDICION_DIA_HABIL, CORTE_OBSERVACION)
display(sens); sens.to_csv(f"{CARPETA_SALIDAS}P3_sensibilidad_sla.csv", index=False)

variante,unidad,cuenta_dia_admision,regla_red,guias,prevalencia,pct_vencido_antes_de_t0,sin_vencimiento
0,horas continuas (regla v2),false,ACTIVA_MAS_RECIENTE,726170,0.7880175388253756,0.5310905159948772,0
1,días hábiles,false,ACTIVA_MAS_RECIENTE,726170,0.45250867650003335,0.40183152705289393,0
2,días hábiles,true,ACTIVA_MAS_RECIENTE,726170,0.9293995348266492,0.9224685679661787,0
3,días hábiles,false,MAXIMO,726170,0.45250728603528445,0.4018287728768746,0
4,días hábiles,true,MAXIMO,726170,0.9293995348266492,0.9224685679661787,0


In [0]:
# Balance de registros: de LOG_GUIAS_BOGOTA a la base v3 (solo conteos)
T = f"read_parquet('{RUTA_GOLD_V3}')"
balance = duckdb.sql(f"""
    SELECT 'Guías en LOG_GUIAS_BOGOTA_V4' AS etapa, COUNT(*) AS guias FROM read_parquet('{RUTA_GUIAS}')
    UNION ALL SELECT 'Base v4: con salida a zona urbana posterior a la admisión', COUNT(*) FROM {T}
    UNION ALL SELECT '  cambian de T0 por exigir salida posterior a la admisión', COUNT(*) FILTER (WHERE t0_sin_condicion < t0) FROM {T}
    UNION ALL SELECT '  entrega anterior a la admisión (sin etiqueta)', COUNT(*) FILTER (WHERE estado_desenlace = 'ENTREGA_ANTES_DE_ADMISION') FROM {T}
    UNION ALL SELECT '  abiertas vencidas al corte (etiqueta 1)', COUNT(*) FILTER (WHERE estado_desenlace = 'ABIERTO_VENCIDO') FROM {T}
    UNION ALL SELECT '  abiertas sin vencer al corte (sin etiqueta)', COUNT(*) FILTER (WHERE estado_desenlace = 'ABIERTO_NO_VENCIDO') FROM {T}
    UNION ALL SELECT '  entrega sin hora', COUNT(*) FILTER (WHERE entrega_sin_hora) FROM {T}
    UNION ALL SELECT '  admitidas después del corte', COUNT(*) FILTER (WHERE admitida_despues_del_corte) FROM {T}
    UNION ALL SELECT '  con SLA único', COUNT(*) FROM {T} WHERE origen_sla = 'SLA_UNICO'
    UNION ALL SELECT '  con SLA por regla (clave repetida con plazos distintos)', COUNT(*) FROM {T} WHERE origen_sla = 'SLA_POR_REGLA'
    UNION ALL SELECT '  con SLA por sustitución', COUNT(*) FROM {T} WHERE origen_sla = 'SLA_CON_SUSTITUCION'
    UNION ALL SELECT '  sin SLA', COUNT(*) FROM {T} WHERE origen_sla = 'SIN_SLA'
    UNION ALL SELECT '  devoluciones (aparte)', COUNT(*) FROM {T} WHERE estado_desenlace = 'DEVUELTO'
    UNION ALL SELECT '  siniestros (aparte)', COUNT(*) FROM {T} WHERE estado_desenlace = 'SINIESTRADO'
    UNION ALL SELECT '  con etiqueta identificable', COUNT(incumplimiento_sla) FROM {T}
    UNION ALL SELECT '  ya vencidas antes de T0', COUNT(*) FILTER (WHERE vencido_antes_de_t0) FROM {T}
    UNION ALL SELECT '  última milla negativa (auditar)', COUNT(*) FILTER (WHERE h_ultima_milla < 0) FROM {T}
""").df()
display(balance); balance.to_csv(f"{CARPETA_SALIDAS}P3_balance.csv", index=False)
# Guías de LOG_GUIAS_BOGOTA_V4 que no entran por no tener salida posterior a la admisión
sin_t0 = duckdb.sql(f"""
    WITH G AS (SELECT CAST(NUMERO_GUIA AS BIGINT) AS guia, FECHA_ENVIO FROM read_parquet('{RUTA_GUIAS}')),
    M AS (SELECT CAST(NUMERO_GUIA AS BIGINT) AS guia, ID_PROCESO, FECHA_HORA_MOVIMIENTO FROM read_parquet('{RUTA_MOVS_GLOB}')),
    X AS (SELECT G.guia, bool_or(M.ID_PROCESO IN (5, 6)) AS con_5_6, bool_or(M.ID_PROCESO = {ID_SALIDA_ZONA}) AS con_9,
                 bool_or(M.ID_PROCESO = {ID_SALIDA_ZONA} AND M.FECHA_HORA_MOVIMIENTO >= G.FECHA_ENVIO - INTERVAL ({int(TOLERANCIA_T0_MIN)}) MINUTE) AS con_9_posterior
          FROM G LEFT JOIN M USING (guia) GROUP BY 1)
    SELECT COUNT(*) FILTER (WHERE con_5_6 IS NULL OR (NOT con_5_6 AND NOT con_9)) AS sin_movimientos_5_6_9,
           COUNT(*) FILTER (WHERE con_5_6 AND NOT con_9)                        AS con_5_o_6_sin_salida_9,
           COUNT(*) FILTER (WHERE con_9 AND NOT con_9_posterior)                AS solo_salidas_anteriores_a_la_admision
    FROM X""").df()
display(sin_t0); sin_t0.to_csv(f"{CARPETA_SALIDAS}P3_sin_t0.csv", index=False)

# Documentos repetidos por clave en la fuente (la base se queda con uno por clave)
display(duckdb.sql(f"""SELECT COUNT(*) AS claves_documento_repetidas FROM (
    SELECT ID_PROCESO, NUMERO_MOVIMI, CONSECUT_MOVIMI FROM read_parquet('{RUTA_DOCS_GLOB}')
    GROUP BY ALL HAVING COUNT(*) > 1)""").df())

etapa,guias
Guías en LOG_GUIAS_BOGOTA_V4,832913
Base v4: con salida a zona urbana posterior a la admisión,749884
cambian de T0 por exigir salida posterior a la admisión,149
entrega anterior a la admisión (sin etiqueta),11
abiertas vencidas al corte (etiqueta 1),7714
abiertas sin vencer al corte (sin etiqueta),7059
entrega sin hora,7906
admitidas después del corte,1
con SLA único,739027
con SLA por regla (clave repetida con plazos distintos),3250


sin_movimientos_5_6_9,con_5_o_6_sin_salida_9,solo_salidas_anteriores_a_la_admision
71317,8405,3307


claves_documento_repetidas
3802


In [0]:
# Manifiesto de la corrida: entradas, parámetros, versiones y conteos (la sal NO se guarda)
import json, glob, sys, datetime
def archivos(patron):
    return [{"archivo": f, "bytes": os.path.getsize(f), "modificado": datetime.datetime.fromtimestamp(os.path.getmtime(f)).isoformat()}
            for f in sorted(glob.glob(patron))]
manifiesto = {
    "fecha_corrida": datetime.datetime.now().isoformat(), "python": sys.version.split()[0], "duckdb": duckdb.__version__,
    "entradas": {"guias": archivos(RUTA_GUIAS), "movimientos": archivos(RUTA_MOVS_GLOB), "documentos": archivos(RUTA_DOCS_GLOB),
                 "calendario": archivos(RUTA_CALENDARIO), "parametricas": {k: archivos(v) for k, v in P.items()}},
    "parametros": {"CAMPO_ENTREGA": CAMPO_ENTREGA, "CORTE_OBSERVACION": CORTE_OBSERVACION, "ID_SALIDA_ZONA": ID_SALIDA_ZONA,
                   "TOLERANCIA_T0_MIN": TOLERANCIA_T0_MIN, "MAX_EXTRACCION_GUIAS": str(MAX_EXTRACCION_GUIAS),
                   "MAX_EXTRACCION_MOVS": str(MAX_EXTRACCION_MOVS),
                   "SLA_EN_DIAS_HABILES": SLA_EN_DIAS_HABILES, "CUENTA_DIA_ADMISION": CUENTA_DIA_ADMISION,
                   "REGLA_RED": REGLA_RED, "CONDICION_DIA_HABIL": CONDICION_DIA_HABIL,
                   "ESTADOS_DEVOLUCION": ESTADOS_DEVOLUCION, "ESTADOS_SINIESTRO": ESTADOS_SINIESTRO,
                   "INCUMPLE_SI_IGUAL": INCUMPLE_SI_IGUAL, "seudonimo_guia": "md5(guia || sal), sal en el Volume"},
    "salida": RUTA_GOLD_V3, "control": tablas["control"].to_dict("records"), "balance": balance.to_dict("records"),
    "sensibilidad": sens.to_dict("records")}
with open(f"{CARPETA_SALIDAS}P3_manifiesto.json", "w") as f:
    json.dump(manifiesto, f, indent=2, default=str)
print("Manifiesto guardado")

Manifiesto guardado


## Qué entregar

Los CSV `P3_*` de `V4/` (en especial `P3_control.csv`, `P3_balance.csv`, `P3_sin_t0.csv` y `P3_por_mes.csv`) y `P3_manifiesto.json`. Con los controles aprobados se corre `0160_ClimaAlineado_4` y luego `0411_Preparacion_Modelado_2`.

`por_mes` debe mostrar, en los últimos meses, cuántas guías siguen abiertas (vencidas o no). La ventana final de 0411 (T0 al menos 86,8 h antes del corte) saca las que no tuvieron tiempo de cerrarse.